# Módulo 1: Fundamentos de Probabilidad e Inferencia Bayesiana — Teoría

**Módulo**: `1` | **Tipo**: TEÓRICO | **Fecha**: 2026-05-20

> **Prerequisitos**: Cálculo integral (integrales básicas, cambio de variable), álgebra lineal básica (vectores, multiplicación matricial), Python con NumPy.

## Objetivos de aprendizaje

Al finalizar este módulo el estudiante será capaz de:

1. **Derivar** el Teorema de Bayes desde los axiomas de Kolmogorov y la regla de la cadena, sin omitir pasos.
2. **Distinguir** con precisión los roles del *prior*, la *likelihood*, el *posterior* y la *evidencia* en el ciclo bayesiano.
3. **Aplicar** la actualización bayesiana secuencial en el modelo Beta-Binomial para estimar parámetros discretos.
4. **Calcular** el posterior analítico en el caso conjugado Beta-Binomial y en el caso Gaussiano-Gaussiano.
5. **Interpretar** cómo el prior y la cantidad de datos determinan la "resistencia al cambio" del posterior.

In [1]:
import sys
import numpy as np
from math import comb
import scipy
from scipy import stats
from scipy.integrate import quad

np.random.seed(42)
assert sys.version_info >= (3, 12), f"Python 3.12+ requerido, tienes {sys.version}"
print(f"Python {sys.version_info.major}.{sys.version_info.minor}  |  NumPy {np.__version__}  |  SciPy {scipy.__version__}  ✓")

Python 3.14  |  NumPy 2.4.6  |  SciPy 1.17.1  ✓


## Intuición — ¿Por qué actualizar creencias?

Imagina que un médico evalúa si un paciente tiene una enfermedad rara.
Antes de ver ningún dato, tiene una **creencia inicial** (*prior*): "solo 1 de cada 1000 personas padece esta enfermedad".

Cuando el paciente se hace un test y el resultado es **positivo**, el médico actualiza su creencia combinando:

- La **creencia inicial** (*prior*): prevalencia = 0.1 %
- La **verosimilitud** (*likelihood*) del test: si la persona está enferma, hay 99 % de probabilidad de positivo.

La **creencia actualizada** (*posterior*) es la respuesta a: *¿Qué tan probable es que este paciente esté enfermo, dado que su test fue positivo?*

Sorprendentemente, la respuesta es **menos del 10 %**. ¿Por qué? Porque el *prior* muy bajo domina.
En la siguiente celda verificamos esto numéricamente.

In [2]:
# Ejemplo del médico — verificación numérica de Bayes
P_E    = 0.001   # prior: P(enfermedad)
P_T_E  = 0.99    # sensibilidad: P(test+ | enfermedad)
P_T_nE = 0.01    # 1 - especificidad: P(test+ | sano)

# Evidencia: P(test+) por ley de probabilidad total
P_T = P_T_E * P_E + P_T_nE * (1 - P_E)

# Posterior: P(enfermedad | test+)
P_E_T = (P_T_E * P_E) / P_T

print(f"Prior P(E)              = {P_E:.4f}  ({100*P_E:.1f}%)")
print(f"Evidencia P(T+)         = {P_T:.5f}  ({100*P_T:.3f}%)")
print(f"Posterior P(E | T+)     = {P_E_T:.4f}  ({100*P_E_T:.1f}%)")
print()
print("→ Aunque el test es 99% preciso, solo ~9% de los positivos están enfermos.")
print("→ El prior bajo domina. Bayes nos protege del error de la tasa base.")

Prior P(E)              = 0.0010  (0.1%)
Evidencia P(T+)         = 0.01098  (1.098%)
Posterior P(E | T+)     = 0.0902  (9.0%)

→ Aunque el test es 99% preciso, solo ~9% de los positivos están enfermos.
→ El prior bajo domina. Bayes nos protege del error de la tasa base.


## 1. Probabilidad condicional y regla de la cadena

### Definición

Sea $(\Omega, \mathcal{F}, P)$ un espacio de probabilidad y $A, B \in \mathcal{F}$ eventos con $P(B) > 0$.
La **probabilidad condicional** de $A$ dado $B$ se define como:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

Esta definición es exacta: no necesita intuición geométrica adicional; surge directamente de los axiomas de Kolmogorov.

### Regla de la cadena

Reorganizando la definición obtenemos la **regla de la cadena**:

$$P(A \cap B) = P(A \mid B) \cdot P(B)$$

Aplicando simétricamente para $P(B \mid A)$ (asumiendo $P(A) > 0$):

$$P(A \cap B) = P(B \mid A) \cdot P(A)$$

Por lo tanto:

$$\boxed{P(B \mid A) \cdot P(A) = P(A \mid B) \cdot P(B)}$$

Esta igualdad es el punto de partida para derivar el Teorema de Bayes.

In [3]:
# Verificación numérica de la regla de la cadena
rng = np.random.default_rng(42)
n_sim = 500_000

# Lanzamos un dado justo (valores 1..6)
# Evento A: dado ≤ 3  → P(A) = 1/2
# Evento B: dado par  → P(B) = 1/2
dados = rng.integers(1, 7, size=n_sim)

A = dados <= 3       # {1, 2, 3}
B = dados % 2 == 0   # {2, 4, 6}

P_A    = A.mean()
P_B    = B.mean()
P_AyB  = (A & B).mean()       # A ∩ B = {2}  → P = 1/6
P_A_B  = P_AyB / P_B          # definición P(A|B)
P_B_A  = P_AyB / P_A          # definición P(B|A)

izq = P_A_B * P_B   # debe igualar P(A ∩ B)
der = P_B_A * P_A   # también debe igualar P(A ∩ B)

print(f"P(A)         = {P_A:.4f}  (teórico = 0.5000)")
print(f"P(B)         = {P_B:.4f}  (teórico = 0.5000)")
print(f"P(A ∩ B)     = {P_AyB:.4f}  (teórico = 0.1667 = 1/6)")
print(f"P(A | B)     = {P_A_B:.4f}  (teórico = 0.3333 = 1/3)")
print(f"P(B | A)     = {P_B_A:.4f}  (teórico = 0.3333 = 1/3)")
print()
print(f"P(A|B)·P(B)  = {izq:.6f}")
print(f"P(B|A)·P(A)  = {der:.6f}")
assert abs(izq - der) < 0.002, "Regla de la cadena fallida"
print("✓ Regla de la cadena verificada: P(A|B)·P(B) = P(B|A)·P(A)")

P(A)         = 0.4995  (teórico = 0.5000)
P(B)         = 0.4996  (teórico = 0.5000)
P(A ∩ B)     = 0.1665  (teórico = 0.1667 = 1/6)
P(A | B)     = 0.3332  (teórico = 0.3333 = 1/3)
P(B | A)     = 0.3333  (teórico = 0.3333 = 1/3)

P(A|B)·P(B)  = 0.166494
P(B|A)·P(A)  = 0.166494
✓ Regla de la cadena verificada: P(A|B)·P(B) = P(B|A)·P(A)


## 2. Teorema de Bayes — Derivación desde axiomas

### Paso 1: punto de partida

De la regla de la cadena (sección anterior):

$$P(B \mid A) \cdot P(A) = P(A \mid B) \cdot P(B)$$

### Paso 2: despejar $P(B \mid A)$

Dividimos ambos lados entre $P(A)$ (asumiendo $P(A) > 0$):

$$P(B \mid A) = \frac{P(A \mid B) \cdot P(B)}{P(A)}$$

### Paso 3: expandir el denominador

Por la **ley de probabilidad total**, si $\{B_i\}_{i=1}^n$ es una partición de $\Omega$:

$$P(A) = \sum_{i=1}^{n} P(A \mid B_i) \cdot P(B_i)$$

En el caso binario ($B$ y $\overline{B}$):

$$P(A) = P(A \mid B) \cdot P(B) + P(A \mid \overline{B}) \cdot P(\overline{B})$$

### Paso 4: notación bayesiana

Renombramos: $\theta$ = parámetro (hipótesis), $D$ = datos (evidencia observada):

$$\boxed{P(\theta \mid D) = \frac{P(D \mid \theta) \cdot P(\theta)}{P(D)}, \qquad P(D) = \int P(D \mid \theta)\, P(\theta)\, d\theta}$$

| Término | Nombre técnico | Interpretación |
|---------|---------------|----------------|
| $P(\theta)$ | **Prior** | Creencia inicial sobre $\theta$, antes de ver los datos |
| $P(D \mid \theta)$ | **Likelihood** (verosimilitud) | Compatibilidad de los datos con el valor de $\theta$ |
| $P(\theta \mid D)$ | **Posterior** | Creencia actualizada tras observar $D$ |
| $P(D)$ | **Evidencia** | Constante de normalización |

> **Nota**: $P(\theta \mid D) \propto P(D \mid \theta) \cdot P(\theta)$ — para inferencia relativa no necesitamos $P(D)$.

In [4]:
# Verificación del Teorema de Bayes con el ejemplo del médico
P_E    = 0.001   # prior P(enfermedad)
P_T_E  = 0.99    # P(test+ | enfermedad)
P_T_nE = 0.01    # P(test+ | sano)

# Paso 3: evidencia (ley de probabilidad total)
P_T = P_T_E * P_E + P_T_nE * (1 - P_E)

# Paso 4: posteriors para E y para ¬E
P_E_T  = (P_T_E  * P_E)       / P_T
P_nE_T = (P_T_nE * (1 - P_E)) / P_T

print(f"P(E | T+)       = {P_E_T:.6f}")
print(f"P(¬E | T+)      = {P_nE_T:.6f}")
print(f"Suma (debe = 1) = {P_E_T + P_nE_T:.10f}")
assert abs(P_E_T + P_nE_T - 1.0) < 1e-10, "Posterior no normalizado"
print("✓ Teorema de Bayes verificado — posterior correctamente normalizado.")

P(E | T+)       = 0.090164
P(¬E | T+)      = 0.909836
Suma (debe = 1) = 1.0000000000
✓ Teorema de Bayes verificado — posterior correctamente normalizado.


## 3. Distribuciones discretas — Bernoulli y Binomial

### Bernoulli

Una variable $X$ sigue una distribución **Bernoulli** con parámetro $\theta \in [0,1]$ si:

$$P(X = x \mid \theta) = \theta^{x}(1 - \theta)^{1-x}, \quad x \in \{0, 1\}$$

- $\mathbb{E}[X] = \theta$
- $\mathrm{Var}(X) = \theta(1-\theta)$

**Interpretación**: $X=1$ es "éxito" (p. ej., cara en lanzamiento de moneda), $X=0$ es "fracaso". $\theta$ es la probabilidad de éxito.

### Binomial

Si realizamos $n$ ensayos de Bernoulli independientes con la misma $\theta$, el número total de éxitos $X = \sum_{i=1}^n X_i$ sigue:

$$P(X = k \mid n, \theta) = \binom{n}{k} \theta^k (1-\theta)^{n-k}, \quad k \in \{0, 1, \ldots, n\}$$

- $\mathbb{E}[X] = n\theta$
- $\mathrm{Var}(X) = n\theta(1-\theta)$

El coeficiente $\binom{n}{k} = \frac{n!}{k!(n-k)!}$ cuenta las combinaciones de obtener exactamente $k$ éxitos en $n$ intentos.

In [5]:
# Distribución Bernoulli y Binomial — cálculo manual vs scipy
theta = 0.7   # probabilidad de éxito

print("=== Bernoulli(θ=0.7) ===")
for x in (0, 1):
    p = theta**x * (1 - theta)**(1 - x)
    print(f"  P(X={x} | θ=0.7) = {p:.4f}")

print()
print("=== Binomial(n=10, θ=0.7) — comparación manual vs scipy ===")
n_binom = 10
for k in range(n_binom + 1):
    p_manual = comb(n_binom, k) * theta**k * (1 - theta)**(n_binom - k)
    p_scipy  = stats.binom.pmf(k, n_binom, theta)
    assert abs(p_manual - p_scipy) < 1e-12, f"Discrepancia en k={k}"
    if k in (0, 5, 7, 10):
        print(f"  P(X={k:2d} | n=10, θ=0.7) = {p_manual:.6f}")

# Verificar que la suma es 1
total = sum(comb(n_binom, k) * theta**k * (1 - theta)**(n_binom - k) for k in range(n_binom + 1))
assert abs(total - 1.0) < 1e-10, "PMF no normalizada"
print()
print(f"E[X]   = n·θ       = {n_binom * theta:.2f}")
print(f"Var(X) = n·θ·(1-θ) = {n_binom * theta * (1 - theta):.2f}")
print("✓ Distribución Binomial verificada.")

=== Bernoulli(θ=0.7) ===
  P(X=0 | θ=0.7) = 0.3000
  P(X=1 | θ=0.7) = 0.7000

=== Binomial(n=10, θ=0.7) — comparación manual vs scipy ===
  P(X= 0 | n=10, θ=0.7) = 0.000006
  P(X= 5 | n=10, θ=0.7) = 0.102919
  P(X= 7 | n=10, θ=0.7) = 0.266828
  P(X=10 | n=10, θ=0.7) = 0.028248

E[X]   = n·θ       = 7.00
Var(X) = n·θ·(1-θ) = 2.10
✓ Distribución Binomial verificada.


## 4. Variables aleatorias continuas — Gaussiana

### Función de densidad de probabilidad (PDF)

Una variable $X$ sigue una distribución **Normal** (Gaussiana) con media $\mu$ y varianza $\sigma^2 > 0$, denotada $X \sim \mathcal{N}(\mu, \sigma^2)$, si:

$$f(x \mid \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(x - \mu)^2}{2\sigma^2}\right), \quad x \in \mathbb{R}$$

### Propiedades clave

| Propiedad | Valor |
|-----------|-------|
| $\mathbb{E}[X]$ | $\mu$ |
| $\mathrm{Var}(X)$ | $\sigma^2$ |
| $P(\mu - \sigma \leq X \leq \mu + \sigma)$ | $\approx 68.3\%$ |
| $P(\mu - 2\sigma \leq X \leq \mu + 2\sigma)$ | $\approx 95.4\%$ |
| $P(\mu - 3\sigma \leq X \leq \mu + 3\sigma)$ | $\approx 99.7\%$ |

### Linealidad y cierre bajo suma

- Si $X \sim \mathcal{N}(\mu, \sigma^2)$ y $Y = aX + b$, entonces $Y \sim \mathcal{N}(a\mu + b,\; a^2\sigma^2)$.
- Si $X \sim \mathcal{N}(\mu_1, \sigma_1^2)$ y $Y \sim \mathcal{N}(\mu_2, \sigma_2^2)$ son **independientes**, entonces $X + Y \sim \mathcal{N}(\mu_1 + \mu_2,\; \sigma_1^2 + \sigma_2^2)$.

Estas propiedades hacen que la Gaussiana sea la distribución central de los filtros bayesianos lineales.

In [6]:
# Distribución Gaussiana — PDF manual y propiedades
mu, sigma2 = 0.0, 1.0
sigma = np.sqrt(sigma2)

def gaussiana_pdf(x: np.ndarray, mu: float, sigma2: float) -> np.ndarray:
    # PDF de N(mu, sigma^2) calculada manualmente
    return (1.0 / np.sqrt(2 * np.pi * sigma2)) * np.exp(-0.5 * (x - mu)**2 / sigma2)

x_vals = np.linspace(-4, 4, 2000)
pdf_manual = gaussiana_pdf(x_vals, mu, sigma2)
pdf_scipy  = stats.norm.pdf(x_vals, mu, sigma)

# Las dos implementaciones deben coincidir
assert np.allclose(pdf_manual, pdf_scipy, atol=1e-12)
print(f"PDF N(μ={mu}, σ²={sigma2})")
print(f"  f(0)   = {gaussiana_pdf(np.array([0.0]), mu, sigma2)[0]:.6f}  (teórico = 1/√(2π) ≈ 0.3989)")
print(f"  f(1)   = {gaussiana_pdf(np.array([1.0]), mu, sigma2)[0]:.6f}")
print(f"  f(-1)  = {gaussiana_pdf(np.array([-1.0]), mu, sigma2)[0]:.6f}")

# Normalización numérica
integral, err = quad(lambda x: float(gaussiana_pdf(np.array([x]), mu, sigma2)[0]), -np.inf, np.inf)
print(f"\n∫f(x)dx = {integral:.8f}  (debe ser 1.0)")
assert abs(integral - 1.0) < 1e-6

# Regla 68-95-99.7
for k in (1, 2, 3):
    p = stats.norm.cdf(mu + k*sigma, mu, sigma) - stats.norm.cdf(mu - k*sigma, mu, sigma)
    print(f"  P(μ ± {k}σ) = {100*p:.2f}%")
print("✓ Gaussiana verificada.")

PDF N(μ=0.0, σ²=1.0)
  f(0)   = 0.398942  (teórico = 1/√(2π) ≈ 0.3989)
  f(1)   = 0.241971
  f(-1)  = 0.241971

∫f(x)dx = 1.00000000  (debe ser 1.0)
  P(μ ± 1σ) = 68.27%
  P(μ ± 2σ) = 95.45%
  P(μ ± 3σ) = 99.73%
✓ Gaussiana verificada.


## 5. Inferencia bayesiana — Ciclo *prior* → *likelihood* → *posterior*

### El ciclo de actualización

El proceso bayesiano actualiza creencias de forma **incremental**. Sea $\theta$ el parámetro desconocido y $x_1, x_2, \ldots, x_t$ una secuencia de observaciones.

**Inicialización**: Definimos el prior $p(\theta)$.

**Para cada nueva observación $x_t$**:

$$p(\theta \mid x_{1:t}) = \frac{p(x_t \mid \theta) \cdot p(\theta \mid x_{1:t-1})}{p(x_t \mid x_{1:t-1})} \propto p(x_t \mid \theta) \cdot p(\theta \mid x_{1:t-1})$$

El **posterior actual** se convierte en el nuevo prior para la siguiente observación.

### Modelo Beta-Binomial (familia conjugada)

Para estimar la probabilidad de cara $\theta$ de una moneda sesgada:

- **Prior**: $\theta \sim \mathrm{Beta}(\alpha, \beta)$, con densidad $p(\theta) \propto \theta^{\alpha-1}(1-\theta)^{\beta-1}$, $\theta \in [0,1]$
- **Likelihood**: observar $k$ caras en $n$ lanzamientos → $P(k \mid n, \theta) \propto \theta^k(1-\theta)^{n-k}$

**Derivación del posterior** — multiplicando prior × likelihood:

$$p(\theta \mid k) \propto \underbrace{\theta^k(1-\theta)^{n-k}}_{\text{likelihood}} \cdot \underbrace{\theta^{\alpha-1}(1-\theta)^{\beta-1}}_{\text{prior}} = \theta^{(\alpha+k)-1}(1-\theta)^{(\beta+n-k)-1}$$

Reconocemos la forma de una distribución Beta, por lo que:

$$\boxed{\theta \mid k \sim \mathrm{Beta}(\alpha + k,\; \beta + n - k)}$$

**Regla de actualización secuencial** (observación $x_t \in \{0,1\}$):
- $x_t = 1$ (cara): $\alpha \leftarrow \alpha + 1$
- $x_t = 0$ (sol): $\beta \leftarrow \beta + 1$

**Estadísticos de la Beta**:
- Media: $\mathbb{E}[\theta] = \dfrac{\alpha}{\alpha + \beta}$
- Moda: $\dfrac{\alpha - 1}{\alpha + \beta - 2}$ (solo válida para $\alpha, \beta > 1$)
- Varianza: $\dfrac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}$

## Ejemplo numérico paso a paso

Aplicamos el ciclo prior → likelihood → posterior al problema de estimar el sesgo $\theta$ de una moneda, usando el modelo Beta-Binomial. Cada línea de la tabla siguiente muestra el estado de la creencia tras una nueva observación.


In [7]:
# Actualización bayesiana secuencial — Beta-Binomial
alpha0, beta0 = 2.0, 2.0   # prior: Beta(2, 2) — difuso, levemente centrado en 0.5
theta_real    = 0.7          # sesgo real (desconocido para el estimador)

rng = np.random.default_rng(42)
n_obs = 20
obs = rng.binomial(1, theta_real, size=n_obs)   # 1=cara, 0=sol

print(f"Theta real (desconocido): {theta_real}")
print(f"Observaciones: {obs}")
print(f"Caras: {obs.sum()}   Soles: {(1 - obs).sum()}")
print()
print(f"{'t':>3} | {'obs':>3} | {'α':>6} | {'β':>6} | {'E[θ|datos]':>12} | {'Mo[θ|datos]':>12}")
print("-" * 55)

alpha, beta = alpha0, beta0
for t, x in enumerate(obs, start=1):
    alpha += x
    beta  += (1 - x)
    E_theta  = alpha / (alpha + beta)
    Mo_theta = (alpha - 1) / (alpha + beta - 2) if alpha > 1 and beta > 1 else float("nan")
    if t <= 5 or t == n_obs:
        print(f"{t:>3} | {int(x):>3} | {alpha:>6.1f} | {beta:>6.1f} | {E_theta:>12.4f} | {Mo_theta:>12.4f}")
    elif t == 6:
        print(f"{'..':>3} | {'..':>3} | {'..':>6} | {'..':>6} | {'..':>12} | {'..':>12}")

print()
posterior_scipy_mean = stats.beta.mean(alpha, beta)
assert abs(alpha / (alpha + beta) - posterior_scipy_mean) < 1e-10
print(f"Posterior final: θ ~ Beta({alpha:.0f}, {beta:.0f})")
print(f"Media posterior E[θ|datos] = {alpha/(alpha+beta):.4f}  (theta real = {theta_real})")
print("✓ Actualización Beta-Binomial verificada contra scipy.")

Theta real (desconocido): 0.7
Observaciones: [0 1 0 1 1 0 0 0 1 1 1 0 1 0 1 1 1 1 0 1]
Caras: 12   Soles: 8

  t | obs |      α |      β |   E[θ|datos] |  Mo[θ|datos]
-------------------------------------------------------
  1 |   0 |    2.0 |    3.0 |       0.4000 |       0.3333
  2 |   1 |    3.0 |    3.0 |       0.5000 |       0.5000
  3 |   0 |    3.0 |    4.0 |       0.4286 |       0.4000
  4 |   1 |    4.0 |    4.0 |       0.5000 |       0.5000
  5 |   1 |    5.0 |    4.0 |       0.5556 |       0.5714
 .. |  .. |     .. |     .. |           .. |           ..
 20 |   1 |   14.0 |   10.0 |       0.5833 |       0.5909

Posterior final: θ ~ Beta(14, 10)
Media posterior E[θ|datos] = 0.5833  (theta real = 0.7)
✓ Actualización Beta-Binomial verificada contra scipy.


## 6. Caso conjugado Gaussiano

### Problema

Queremos estimar la **media desconocida** $\mu$ de un proceso Gaussiano con varianza $\sigma^2$ **conocida**, dado $n$ observaciones $x_1, \ldots, x_n$.

**Prior**: $\mu \sim \mathcal{N}(\mu_0, \sigma_0^2)$

**Likelihood**: $x_i \mid \mu \sim \mathcal{N}(\mu, \sigma^2)$ — varianza conocida, media desconocida.

### Derivación paso a paso

Usamos la notación de **precisión**: $\tau = 1/\sigma^2$ (inverso de varianza).

**Paso 1** — Log posterior (omitiendo constantes en $\mu$):

$$\ln p(\mu \mid x_{1:n}) = -\frac{(\mu - \mu_0)^2}{2\sigma_0^2} - \sum_{i=1}^n \frac{(x_i - \mu)^2}{2\sigma^2} + \mathrm{const}$$

**Paso 2** — Expandir la suma del likelihood:

$$-\sum_{i=1}^n \frac{(x_i - \mu)^2}{2\sigma^2} = \frac{-n\mu^2 + 2n\bar{x}\mu}{2\sigma^2} + \mathrm{const}, \quad \bar{x} = \tfrac{1}{n}\sum_i x_i$$

**Paso 3** — Agrupar términos en $\mu^2$ y en $\mu$ (completar el cuadrado):

$$\ln p(\mu \mid x_{1:n}) \propto -\frac{1}{2}\underbrace{\left(\tau_0 + n\tau\right)}_{\tau_n}\mu^2 + \underbrace{\left(\tau_0\mu_0 + n\tau\bar{x}\right)}_{=\,\tau_n\mu_n}\mu$$

**Paso 4** — Identificar la forma $-\frac{\tau_n}{2}(\mu - \mu_n)^2$ (Gaussiana en $\mu$):

$$\tau_n = \tau_0 + n\tau = \frac{1}{\sigma_0^2} + \frac{n}{\sigma^2}$$

$$\mu_n = \frac{\tau_0\,\mu_0 + n\tau\,\bar{x}}{\tau_n}$$

**Resultado**:

$$\boxed{\mu \mid x_{1:n} \sim \mathcal{N}\!\left(\mu_n,\; \sigma_n^2 = \frac{1}{\tau_n}\right)}$$

**Interpretación**: Con $n \to \infty$, $\mu_n \to \bar{x}$ (prior olvidado). Con $\sigma_0^2 \to \infty$ (prior muy difuso), $\mu_n \approx \bar{x}$ desde el primer dato.

In [8]:
# Caso conjugado Gaussiano — verificación numérica
mu0     = 0.0    # prior: media
sigma2_0 = 1.0   # prior: varianza (moderadamente informativo)
sigma2   = 0.25  # varianza del proceso de medición (conocida)
mu_real  = 0.8   # media real (desconocida para el estimador)

rng = np.random.default_rng(42)
n_datos = 10
datos = rng.normal(mu_real, np.sqrt(sigma2), size=n_datos)

# Actualización analítica (Paso 3 y 4)
tau0 = 1.0 / sigma2_0
tau  = 1.0 / sigma2
tau_n = tau0 + n_datos * tau
sigma2_n = 1.0 / tau_n
mu_n = (tau0 * mu0 + tau * n_datos * datos.mean()) / tau_n

ic_lo = mu_n - 1.96 * np.sqrt(sigma2_n)
ic_hi = mu_n + 1.96 * np.sqrt(sigma2_n)

print(f"Prior: μ ~ N({mu0}, {sigma2_0})")
print(f"Likelihood: σ² = {sigma2} (conocida)")
print(f"Datos: n={n_datos}, x̄ = {datos.mean():.4f}")
print()
print(f"Precisión prior τ₀      = {tau0:.4f}")
print(f"Precisión por obs τ     = {tau:.4f}")
print(f"Precisión posterior τₙ  = {tau_n:.4f}")
print(f"Media posterior μₙ      = {mu_n:.6f}")
print(f"Varianza posterior σₙ²  = {sigma2_n:.6f}")
print(f"μ real                  = {mu_real}")
print(f"IC 95%: ({ic_lo:.4f}, {ic_hi:.4f})  — contiene μ_real: {ic_lo <= mu_real <= ic_hi}")
print()

# Verificar contra scipy
from scipy.stats import norm as scipy_norm
posterior = scipy_norm(loc=mu_n, scale=np.sqrt(sigma2_n))
assert abs(mu_n - posterior.mean()) < 1e-10
print("✓ Caso conjugado Gaussiano verificado.")

Prior: μ ~ N(0.0, 1.0)
Likelihood: σ² = 0.25 (conocida)
Datos: n=10, x̄ = 0.6322

Precisión prior τ₀      = 1.0000
Precisión por obs τ     = 4.0000
Precisión posterior τₙ  = 41.0000
Media posterior μₙ      = 0.616794
Varianza posterior σₙ²  = 0.024390
μ real                  = 0.8
IC 95%: (0.3107, 0.9229)  — contiene μ_real: True

✓ Caso conjugado Gaussiano verificado.


## Errores comunes

### Error 1: Confundir $P(A \mid B)$ con $P(B \mid A)$ — *Falacia de la transposición*

**Síntoma**: El estudiante usa directamente la sensibilidad del test como el valor predictivo positivo, sin aplicar Bayes. Escribe $P(\text{enfermedad} \mid \text{test+}) = 0.99$ porque "el test tiene 99% de precisión".

**Diagnóstico**: Confunde la *verosimilitud* $P(\text{test+} \mid \text{enfermedad})$ con el *posterior* $P(\text{enfermedad} \mid \text{test+})$. Ambas son iguales solo cuando el prior es uniforme — raramente ocurre en la práctica.

**Solución**: Antes de calcular, identifica explícitamente: (1) ¿cuál es el prior?, (2) ¿cuál es el likelihood?, (3) ¿cuál es la evidencia? Aplica la fórmula de Bayes completa, incluyendo el denominador.

---

### Error 2: Olvidar normalizar el posterior

**Síntoma**: El estudiante computa $p(\theta \mid D) \propto p(D \mid \theta) p(\theta)$ y usa esa cantidad directamente en cálculos de probabilidad, obteniendo valores fuera de $[0, 1]$ o que no suman 1.

**Diagnóstico**: La expresión $p(D \mid \theta) p(\theta)$ es proporcional al posterior, pero no es una distribución válida (no integra/suma a 1). Sin normalizar, todas las probabilidades derivadas son incorrectas.

**Solución**: Divide siempre entre la evidencia $P(D)$ o verifica que tu distribución posterior integra a 1. En familias conjugadas (Beta, Gaussiana), la forma de la distribución garantiza normalización automática.

---

### Error 3: Confundir la distribución Beta como distribución de los datos

**Síntoma**: El estudiante escribe "los lanzamientos siguen $\mathrm{Beta}(\alpha, \beta)$" cuando el prior sobre $\theta$ sigue esa distribución.

**Diagnóstico**: En el modelo Beta-Binomial existen dos niveles: (1) el *parámetro* $\theta \in [0,1]$ sigue $\mathrm{Beta}(\alpha, \beta)$, y (2) los *datos* $X \mid \theta \sim \mathrm{Bernoulli}(\theta)$ toman valores en $\{0,1\}$. Son espacios de probabilidad distintos.

**Solución**: Distingue siempre el *espacio del parámetro* (continuo, $\theta \in [0,1]$) del *espacio de los datos* (discreto, $x \in \{0,1\}$). El prior vive en el espacio del parámetro, no en el de los datos.

## Evaluación

Resuelve los siguientes problemas. Tolerancia numérica aceptable: $\varepsilon = 10^{-4}$.

**Problema M1-E1**: Una moneda tiene prior $\theta \sim \mathrm{Beta}(2, 2)$. Se lanza 10 veces y salen 8 caras.

(a) ¿Cuál es la distribución posterior $p(\theta \mid k=8, n=10)$?
(b) Calcula la media posterior $\mathbb{E}[\theta \mid k=8, n=10]$.
(c) Calcula la moda posterior. ¿Cuál de los dos estadísticos (media o moda) estimas que es más representativo del "mejor estimado puntual" del parámetro?

**Criterio**: $|\hat{\theta}_{\text{tuyo}} - \hat{\theta}_{\text{analítico}}| < 10^{-4}$.

---

**Problema M1-E2**: Sea $\mu \sim \mathcal{N}(0, 4)$ el prior para la media de un proceso con $\sigma^2 = 1$. Se observan 5 datos con media muestral $\bar{x} = 1.5$.

(a) Calcula la precisión posterior $\tau_n$.
(b) Calcula la media posterior $\mu_n$ y la varianza posterior $\sigma_n^2$.
(c) ¿Contiene el intervalo de credibilidad al 95% a $\mu = 1.5$?

---

**Problema M1-E3 (identidad)**: Demuestra numéricamente que si actualizamos el prior $\mathrm{Beta}(3, 5)$ con $n=0$ observaciones, el posterior es idéntico al prior.

## ¿Qué aprendiste?

Aprendimos que…

1. El Teorema de Bayes **no es una fórmula mágica** sino una consecuencia directa de la regla de la cadena de probabilidades. Lo derivamos en cuatro pasos sin apelar a ninguna intuición no justificada.
2. El **prior bajo domina** incluso ante un likelihood fuerte: un test de 99% de precisión puede tener valor predictivo positivo de solo ~9% si la enfermedad es rara (ejemplo del médico).
3. En el modelo **Beta-Binomial**, cada observación actualiza $\alpha$ y $\beta$ con una simple suma — la forma de la distribución se preserva. Esto es la definición de *familia conjugada*.
4. En el modelo **Gaussiano-Gaussiano**, la **precisión posterior** es la suma de las precisiones del prior y las observaciones: $\tau_n = \tau_0 + n/\sigma^2$. Más datos → más precisión → distribución posterior más estrecha.
5. La distinción entre **prior** (creencia previa), **likelihood** (compatibilidad de los datos con un parámetro) y **posterior** (creencia actualizada) es el vocabulario fundamental de todos los filtros bayesianos del curso.

## ¿Qué sigue?

El siguiente módulo es el **Módulo 2: Filtro Bayesiano Discreto**.

Con el ciclo *prior → likelihood → posterior* aprendido aquí, implementaremos un filtro que actualiza una distribución discreta sobre un espacio de estados **dinámico** $x_k$ que cambia en el tiempo. La diferencia clave respecto a este módulo: ya no estimamos un parámetro estático $\theta$, sino un **estado** que evoluciona según una ecuación de transición.

Las ecuaciones de actualización que aprendimos se generalizan a:

$$\underbrace{p(x_k \mid z_{1:k-1})}_{\text{prior del filtro}} = \sum_{x_{k-1}} p(x_k \mid x_{k-1})\; p(x_{k-1} \mid z_{1:k-1}) \quad \text{(predicción)}$$

$$\underbrace{p(x_k \mid z_{1:k})}_{\text{posterior del filtro}} \propto p(z_k \mid x_k)\; p(x_k \mid z_{1:k-1}) \quad \text{(corrección)}$$